# §F — Served LLM+NTK NLL gate

Release-gate check for **NTK Phase-2**: confirm the *served* `ntk_model` ISVC reproduces the controller's hook-attached reference NLL.

It hits a deployed KServe ISVC's OpenAI `/v1/completions` (`echo` + `logprobs=1`) and computes the **same** teacher-forced, completion-masked NLL as the §E `evaluate_nll.py` `_completion_nll` arm — so the served number is apples-to-apples with the (reused) hook-attached `reference`.

The **namespace, endpoint, and served model name all resolve automatically via cogflow** from your user context + the ISVC name (you set only `ISVC_NAME`). `cogflow.serving.list_models` returns the ISVC's `served_model_url`; the served model name is read from the endpoint's `/v1/models`.

**Apples-to-apples by construction:** tokenize `prompt`+`completion` with the SAME base tokenizer and SAME split as the reference, then send the **exact token ids** as the completions `prompt`. vLLM uses them verbatim → no retokenization drift; the completion span is exactly `token_logprobs[len(prompt_ids):]`.

**Gate:** `|served − reference| / reference ≤ threshold` (default 1%). Served ≈ reference is expected (the plugin uses the same `ControllerRuntime.apply` hooks as the reference arm).

Prereqs: the notebook env has cogflow installed + Kubernetes access (kubeconfig) to the namespace the ISVC runs in; the ISVC was deployed **without** `--quantization`; `REFERENCE_NLL` was computed (§E reference arm) on the **same** controller this ISVC serves.


In [ ]:
# !pip install transformers requests cogflow

In [ ]:
# --- Config -------------------------------------------------------------
BASE = "Qwen/Qwen2.5-0.5B-Instruct"          # same base the ISVC serves
EVAL_JSONL = "runs/gsm8k_small/eval.jsonl"   # GSM8K eval split (prepare_dataset.py)

# The ONLY required field: the deployed ntk_model ISVC name. Namespace,
# endpoint, and served model name all auto-resolve from your user context.
ISVC_NAME = "<isvc-name>"                     # the deployed InferenceService name
NAMESPACE = None                              # None -> your user namespace (cogflow common.get_namespace())

# Optional manual overrides (leave None to auto-resolve via cogflow / /v1/models).
ENDPOINT = None                               # e.g. "https://<host>/openai/v1"
SERVED_MODEL_NAME = None                      # the OpenAI "model" field

API_KEY = None                                # only for external-ingress/cross-ns access; in-namespace needs none
REFERENCE_NLL = 0.5888                        # §E hook-attached ref for the SAME controller
THRESHOLD = 0.01                              # |served-reference|/reference pass band
MAX_TOKENS = 0                                # 0 = echo only; auto-falls back to 1 if rejected

In [ ]:
# --- Resolve the ISVC endpoint + served model name via cogflow ----------
import requests
from cogflow import serving as cogflow_serving


def resolve_endpoint(isvc_name, namespace=None):
    """Look up the ISVC via cogflow and return (openai_base_url, processed_isvc)."""
    models = cogflow_serving.list_models(namespace=namespace, isvc_name=isvc_name)
    if not models:
        raise RuntimeError(f"ISVC {isvc_name!r} not found in namespace {namespace!r}")
    m = models[0]
    if m.get("status") != "ready":
        print(f"WARNING: ISVC status = {m.get('status')!r} (expected 'ready')")
    url = m.get("served_model_url")
    if not url:
        raise RuntimeError(
            f"ISVC {isvc_name!r} has no served_model_url yet (status={m.get('status')!r})"
        )
    # KServe's huggingface runtime mounts the OpenAI API under /openai/v1.
    return url.rstrip("/") + "/openai/v1", m


def discover_served_model_name(endpoint, api_key=None):
    """Read the served model id from the endpoint's /v1/models (the name vLLM
    expects in the completions 'model' field)."""
    headers = {"Authorization": f"Bearer {api_key}"} if api_key else {}
    r = requests.get(endpoint.rstrip("/") + "/models", headers=headers, timeout=30)
    r.raise_for_status()
    data = r.json().get("data", [])
    if not data:
        raise RuntimeError("served /v1/models returned no models")
    return data[0]["id"]


if ENDPOINT is None:
    ENDPOINT, _isvc = resolve_endpoint(ISVC_NAME, NAMESPACE)
    print("resolved endpoint    :", ENDPOINT)
if SERVED_MODEL_NAME is None:
    SERVED_MODEL_NAME = discover_served_model_name(ENDPOINT, API_KEY)
print("served model name    :", SERVED_MODEL_NAME)

In [ ]:
import json

def load_examples(path):
    """Read the {prompt, completion} JSONL (same contract as the trainer / §E harness)."""
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def completion_token_split(tokenizer, prompt, completion):
    """Reproduce _completion_nll's split exactly: prompt with special tokens,
    completion with add_special_tokens=False, concatenated. Returns int lists."""
    prompt_ids = tokenizer(prompt).input_ids
    completion_ids = tokenizer(completion, add_special_tokens=False).input_ids
    return prompt_ids, completion_ids, prompt_ids + completion_ids

In [ ]:
def completion_logprobs_from_echo(logprobs_obj, input_ids, completion_start):
    """Slice per-token logprobs for the completion span from an echo response.

    Guards: logprobs_obj must be a dict (a server returning ``logprobs: null``
    would otherwise raise an opaque AttributeError); a None *inside* the
    completion span is a hard error -> never coerce to 0.0 (false PASS)."""
    if not isinstance(logprobs_obj, dict):
        raise RuntimeError(
            f"served response 'logprobs' is missing or not an object "
            f"(got {type(logprobs_obj).__name__}); the server returned no logprobs"
        )
    token_logprobs = logprobs_obj.get("token_logprobs")
    if token_logprobs is None:
        raise RuntimeError("served response has no logprobs.token_logprobs")
    if len(token_logprobs) < len(input_ids):
        raise RuntimeError(
            f"served logprobs length {len(token_logprobs)} < input length "
            f"{len(input_ids)}; cannot align completion span"
        )
    comp = token_logprobs[completion_start:len(input_ids)]
    if any(lp is None for lp in comp):
        raise RuntimeError(
            "served logprobs contain None inside the completion span; "
            "the server did not return a logprob for every completion token"
        )
    return comp


def _post_completion(url, headers, base_body, max_tokens):
    return requests.post(url, headers=headers, json={**base_body, "max_tokens": max_tokens}, timeout=120)


def served_completion_nll(endpoint, served_model_name, api_key, tokenizer, examples, max_tokens=0):
    """Teacher-forced completion-NLL via the served OpenAI endpoint.

    Same aggregation as _completion_nll: global sum of per-completion-token
    negative logprob / total completion tokens (NOT mean-of-means).
    max_tokens fallback: some vLLM builds reject max_tokens=0 with echo; on a
    400 we retry once with max_tokens=1 (trailing token sliced off) and use 1
    for the rest of the run."""
    url = endpoint.rstrip("/") + "/completions"
    headers = {"Content-Type": "application/json"}
    if api_key:
        headers["Authorization"] = f"Bearer {api_key}"

    eff = max_tokens
    total_neg_logprob, total_tokens = 0.0, 0
    for i, ex in enumerate(examples):
        prompt_ids, completion_ids, input_ids = completion_token_split(
            tokenizer, ex["prompt"], ex["completion"])
        if not completion_ids:
            continue
        base_body = {
            "model": served_model_name,
            "prompt": input_ids,   # token ids -> used verbatim, no retokenization
            "echo": True,
            "logprobs": 1,
            "temperature": 0,
        }
        resp = _post_completion(url, headers, base_body, eff)
        if resp.status_code == 400 and eff == 0:
            eff = 1
            resp = _post_completion(url, headers, base_body, eff)
        resp.raise_for_status()
        choice = resp.json()["choices"][0]
        comp = completion_logprobs_from_echo(choice["logprobs"], input_ids, len(prompt_ids))
        total_neg_logprob += -float(sum(comp))
        total_tokens += len(comp)
        if (i + 1) % 10 == 0:
            print(f"[served] {i + 1} examples scored")

    if total_tokens == 0:
        raise RuntimeError("no completion tokens scored")
    return total_neg_logprob / total_tokens

In [ ]:
import math
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE)
examples = load_examples(EVAL_JSONL)
print(f"scoring {len(examples)} examples against {ENDPOINT} (model={SERVED_MODEL_NAME})")

served_nll = served_completion_nll(
    ENDPOINT, SERVED_MODEL_NAME, API_KEY, tokenizer, examples, max_tokens=MAX_TOKENS)

print("\n================== §F served-arm summary ==================")
print(f"served (LLM+NTK ISVC)       : {served_nll:.4f}  ({math.exp(served_nll):.4f} ppl)")
rel = abs(served_nll - REFERENCE_NLL) / max(REFERENCE_NLL, 1e-9)
verdict = "PASS" if rel <= THRESHOLD else "FAIL"
print(f"reference (reused §E hooks)  : {REFERENCE_NLL:.4f}  "
      f"(Δ = {served_nll - REFERENCE_NLL:+.4f}, relative = {rel*100:.2f}%, "
      f"threshold {THRESHOLD*100:.0f}%, {verdict})")

## Notes

- **Endpoint resolution needs cluster access.** `cogflow.serving.list_models` reads the ISVC CRD via the Kubernetes API, so the notebook env needs a kubeconfig for the ISVC's namespace. `served_model_url` is usually the cluster/ingress URL — if it isn't reachable from where you run the notebook, set `ENDPOINT` manually to the externally-reachable URL.
- **Auth:** an in-namespace run against the cluster `served_model_url` needs **no token** — the namespace's Istio AuthorizationPolicy trusts same-namespace traffic (namespace is the auth boundary). `API_KEY` is only for the external ingress or cross-namespace access (Dex session / `kubeflow-userid`).
- **Reference must match the served controller.** `REFERENCE_NLL` is only valid if computed (§E reference arm) on the exact `controller.pt` this ISVC serves.
- **No quantization on the gate ISVC** — served (bf16/fp16) vs reference (fp32) should differ only by float noise (the 1% band). A bigger gap is a serving-path bug to investigate, not a band to widen.
- **Spot-check first:** on one example, the negated per-token logprobs here should match the reference arm's per-token cross-entropy within fp tolerance before trusting the aggregate.
